# Deep Learning
<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/marcinsawinski/UEP_KIE_DL_CODE2024/blob/main/dl03_tuning.ipynb" target="_parent">
      <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
    </a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marcinsawinski/UEP_KIE_DL_CODE2024/blob/main/dl03_tuning.ipynb">
      <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/>
    </a>
  </td>
  <td>
    <a target="_blank" href="https://studiolab.sagemaker.aws/import/github/marcinsawinski/UEP_KIE_DL_CODE2024/blob/main/dl03_tuning.ipynb">
      <img src="https://studiolab.sagemaker.aws/studiolab.svg" alt="Open in SageMaker Studio Lab"/>
    </a>
  </td>
</table>

# Tasks
1. Install and connect to [WandB](https://wandb.ai) and Run a training simulation
2. Build NN model for classification on Fashion MNIST dataset. Log training to WandB in own project.
3. Create hyperparameter search with WandB sweep.
4. Experiment with Optimizers (SGD, ADAM) and optimizer parameters - number of epochs, learnign rate.
5. Experiment with network depth and number of neurons in layers
6. Experiment with Schedulers
7. Experiment with Dropout layers
8. Experiment with Batch Normalization layers
9. Try early stopping.
10. Find optimal setup. Retrain with the optimal setup and Log training to WandB in project called DL25-FMNIST


## Task1 - Install and connect to [WandB](https://wandb.ai)
1. Run `pip install wandb -qU`in terminal or `!pip install wandb -qU` in jupyter cell
2. Login using `wandb login` (terminal) or !wandb login' (jupyter cell). Alternatively you can run
```
    import wandb
    wandb.login()
```
3. Update wandb.init code with your own project name e.g. student_surname_firstname. The entity can be "uep-kie-dl25" if you are already assigned.
4. Run simulation and review results on the [WandB](https://wandb.ai) website.


You will find WandB detailed example here:
https://colab.research.google.com/github/wandb/examples/blob/master/colabs/intro/Intro_to_Weights_%26_Biases.ipynb


In [1]:
!pip install wandb -qU

In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 84872 (UEP-DL26) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# your code for installation and login goes here

In [4]:
import random
import time
import wandb
user = "biskup_franciszek"
cfg = {
    "learning_rate": 0.02,
    "architecture": "ANN",
    "dataset": "dummy",
    "epochs": 5,
}

name = f"{cfg['architecture']}_{cfg['dataset']}_lr{cfg['learning_rate']}_ep{cfg['epochs']}_{time.strftime('%m%d-%H%M')}"
project = (f"student_{user}_dummy")
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="UEP-DL26",
    # Set the wandb project where this run will be logged.
    project=project,
    # Track hyperparameters and run metadata.
    name=name,
    config=cfg,
)


# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})

# Finish the run and upload any remaining data.
run.finish()

acc,▁▂▂█▅▆▇█
loss,█▅▁▂▂▁▁▁
acc,0.92641
loss,0.14805


# Task 2 - Build NN model for classification on Fashion MNIST dataset. Log training to WandB in own project.

Create a basic NN model:
- Flatten data with `nn.Flatten()` layer
- 3 linear layers with 784, 128 and 64 inputs. outputs are 128,64 and 10 e.g. `nn.Linear(64, 10)`
- ReLU activation `nn.ReLU()`  (no activation for output layer)
- use cross entropy loss as criterion `nn.CrossEntropyLoss()`
- use  SGD optimizer `optim.SGD(model.parameters(), lr=config.lr)`


https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Organizing_Hyperparameter_Sweeps_in_PyTorch_with_W%26B.ipynb

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math
import time
import wandb

In [6]:
cfg = {
    "learning_rate": 0.02,
    "architecture": "ANN",
    "dataset": "FMNIST",
    "epochs": 5,
    "batch_size": 32,
    "lr": 1e-3,
}
name = f"{cfg['architecture']}_{cfg['dataset']}_lr{cfg['learning_rate']}_ep{cfg['epochs']}_{time.strftime('%m%d-%H%M')}"
project = f"student_{user}_FMNIST"

In [8]:
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="UEP-DL26",
    # Set the wandb project where this run will be logged.
    project=project,
    # Track hyperparameters and run metadata.
    name=name,
    config=cfg,
)

config = wandb.config

# 1. Device configuration
device = torch.device("mps" if torch.cuda.is_available() else "cpu")

# 2. Transformations
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)

# 3. Load the dataset
train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size * 4)
n_steps_per_epoch = math.ceil(len(train_loader.dataset) / wandb.config.batch_size)


# 4. Define the model
class FashionClassifier(nn.Module):
    def __init__(self):
        super(FashionClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )


    def forward(self, x):
        return self.network(x)


model = FashionClassifier().to(device)

# 5. Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=config.lr)

# 6. Training loop
num_epochs = config.epochs
example_ct = 0
step_ct = 0

for epoch in range(num_epochs):
    model.train()
    for step, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        example_ct += len(images)
        train_step_ct = (step + 1 + (n_steps_per_epoch * epoch)) / n_steps_per_epoch
        log_step_ct = step + 1 + (n_steps_per_epoch * epoch)
        metrics = {
            "train/train_loss": loss,
            "train/epoch": train_step_ct,
            "train/example_ct": example_ct,
        }
        # print(log_step_ct)
        if step + 1 < n_steps_per_epoch:
            # Log train metrics to wandb
            wandb.log(metrics)

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

    # 7. Evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"Test Accuracy: {100 * correct / total:.2f}%")
    metrics = {
        "test/epoch": epoch + 1,
        "test/accuracy": 100 * correct / total,
    }
    wandb.log(metrics)
wandb.finish()

100%|██████████| 26.4M/26.4M [00:01<00:00, 17.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 280kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.20MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.26MB/s]


Epoch [1/5], Loss: 1.2609
Test Accuracy: 60.64%
Epoch [2/5], Loss: 0.7825
Test Accuracy: 71.82%
Epoch [3/5], Loss: 0.6603
Test Accuracy: 73.91%
Epoch [4/5], Loss: 0.7982
Test Accuracy: 75.66%
Epoch [5/5], Loss: 0.6350
Test Accuracy: 77.56%


test/accuracy,▁▆▆▇█
test/epoch,▁▃▅▆█
train/epoch,▁▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇███
train/example_ct,▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇████
train/train_loss,███▆▆▆▅▅▄▄▄▄▃▂▄▂▂▄▃▃▂▂▂▃▂▂▁▂▂▂▁▁▁▃▁▂▂▁▁▃
test/accuracy,77.56
test/epoch,5
train/epoch,4.99947
train/example_ct,299968
train/train_loss,0.70327


# Task 3 - Create hyperparameter search with WandB sweep.
Follow instructions available from WandB

https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Organizing_Hyperparameter_Sweeps_in_PyTorch_with_W%26B.ipynb

In [ ]:
# Experiment with sweeps

In [40]:
sweep_config = {
    'method': 'random'
    }

In [52]:
metric = {
    'name': 'loss',
    'goal': 'minimize'
    }

sweep_config['metric'] = metric

In [32]:
cfg = {
    "learning_rate": 0.02,
    "architecture": "ANN",
    "dataset": "FMNIST",
    "epochs": 5,
    "batch_size": 32,
    "lr": 1e-3,
}
name = f"{cfg['architecture']}_{cfg['dataset']}_lr{cfg['learning_rate']}_ep{cfg['epochs']}_{time.strftime('%m%d-%H%M')}"
project = f"student_{user}_FMNIST"

In [60]:
parameters_dict = {
    'optimizer': {'values': ['adam', 'sgd']
},
    'fc_layer_size': {
        'values': [128, 256, 512]
        },
    'dropout': {
          'values': [0.0, 0.3, 0.5]
        },
    'batch_size': {
        #'distribution': 'q_log_uniform_values',
        #'max': 256, 'min': 32, 'q': 8
        'values': [32, 64, 128]},
    'learning_rate': {
        'distribution': 'log_uniform_values',
        'min': 0.0001,
        'max': 0.1
      },
    'use_batchnorm': {'values': [True, False]},
    }

sweep_config['parameters'] = parameters_dict

In [61]:
parameters_dict.update({
    'epochs': {
        'value': 20}
    })

parameters_dict.update({
    'scheduler': {
        'values': ['step', 'none']}
    })

In [62]:
parameters_dict.update({
    'learning_rate': {
        # a flat distribution between 0 and 0.1
        'distribution': 'uniform',
        'min': 0,
        'max': 0.1
      },
    'batch_size': {
        # integers between 32 and 256
        # with evenly-distributed logarithms
        'distribution': 'q_log_uniform_values',
        'q': 8,
        'min': 32,
        'max': 256,
      }
    })

In [63]:
import pprint
pprint.pprint(sweep_config)

{'method': 'random',
 'metric': {'goal': 'minimize', 'name': 'loss'},
 'parameters': {'batch_size': {'distribution': 'q_log_uniform_values',
                               'max': 256,
                               'min': 32,
                               'q': 8},
                'dropout': {'values': [0.0, 0.3, 0.5]},
                'epochs': {'value': 20},
                'fc_layer_size': {'values': [128, 256, 512]},
                'learning_rate': {'distribution': 'uniform',
                                  'max': 0.1,
                                  'min': 0},
                'optimizer': {'values': ['adam', 'sgd']},
                'scheduler': {'values': ['step', 'none']},
                'use_batchnorm': {'values': [True, False]}}}


In [64]:
def build_dataset(batch_size):

    transform = transforms.Compose(
        [transforms.ToTensor(),
        # transforms.Normalize((0.1307,), (0.3081,))
        transforms.Normalize((0.5,), (0.5,))
         ])
    # download MNIST training dataset
    dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
    #sub_dataset = torch.utils.data.Subset(dataset, indices=range(0, len(dataset), 5))

    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size) # zmien na subset aby przyspieszyc

    return loader


def build_network(fc_layer_size, dropout):
    network = nn.Sequential(  # fully-connected, single hidden layer
        nn.Flatten(),
        nn.Linear(784, fc_layer_size), nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(fc_layer_size, 10),
        nn.LogSoftmax(dim=1))

    return network.to(device)


def build_optimizer(network, optimizer, learning_rate):
    if optimizer == "sgd":
        optimizer = optim.SGD(network.parameters(),
                              lr=learning_rate, momentum=0.9)
    elif optimizer == "adam":
        optimizer = optim.Adam(network.parameters(),
                               lr=learning_rate)
    return optimizer


def train_epoch(network, loader, optimizer):
    cumu_loss = 0
    for _, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()

        # ➡ Forward pass
        loss = F.nll_loss(network(data), target)
        cumu_loss += loss.item()

        # ⬅ Backward pass + weight update
        loss.backward()
        optimizer.step()

        wandb.log({"batch loss": loss.item()})

    return cumu_loss / len(loader)

In [65]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from torchvision import datasets, transforms

def train(config=None):
    # Initialize a new wandb run
    with wandb.init(config=config):
        # If called by wandb.agent, as below,
        # this config will be set by Sweep Controller
        config = wandb.config

        loader = build_dataset(config.batch_size)
        network = build_network(config.fc_layer_size, config.dropout)
        optimizer = build_optimizer(network, config.optimizer, config.learning_rate)

        for epoch in range(config.epochs):
            avg_loss = train_epoch(network, loader, optimizer)
            wandb.log({"loss": avg_loss, "epoch": epoch})

In [66]:
sweep_id = wandb.sweep(sweep_config, project=project)
sweep_id

Create sweep with ID: ldtw2kno
Sweep URL: https://wandb.ai/UEP-DL26/student_biskup_franciszek_FMNIST/sweeps/ldtw2kno


'ldtw2kno'

In [ ]:
wandb.agent(sweep_id, train, count=10)

wandb: Agent Starting Run: gy74j288 with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.5
wandb: 	epochs: 20
wandb: 	fc_layer_size: 512
wandb: 	learning_rate: 0.013590672450783247
wandb: 	optimizer: adam
wandb: 	scheduler: none
wandb: 	use_batchnorm: True
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


# Remaining tasks
4. Experiment with Optimizers (SGD, ADAM) and optimizer parameters - number of epochs, learning rate.
5. Experiment with network depth and number of neurons in layers
6. Experiment with Schedulers
7. Experiment with Dropout layers
8. Experiment with Batch Normalization layers
9. Try early stopping.
10. Find optimal setup. Retrain with the optimal setup and Log training to WandB in project called DL25-FMNIST. use your name as job name

**Optimizers**

Available [Optimizers](https://pytorch.org/docs/stable/optim.html):
```import torch.optim as optim
optimizer = optim.SGD(model.parameters(), lr=config.lr)
optimizer = optim.Adam(model.parameters(), lr=config.lr)
```
**Dropout**

Dropout layers: `nn.Dropout(0.5)` - add after the ReLU activations.

**Batch Normalization**

Batch Normalization layers 'nn.BatchNorm1d(64)'  - add right after Linear layers and before the activation functions.


**Scheduler**

```
optimizer = optim.SGD(model.parameters(), lr=0.01)
scheduler = ExponentialLR(optimizer, gamma=0.9)

for epoch in range(20):
    for input, target in dataset:
        optimizer.zero_grad()
        output = model(input)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
    scheduler.step()
```



In [ ]:
# your code here